# Hands-On 2: Build a Multi-Node Workflow

We build a workflow with multiple nodes that pass data through shared state step-by-step.

#### Imports

We import `TypedDict` for the state schema and `StateGraph`, `START`, `END` from LangGraph.

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

#### Define State

Shared state flows through all nodes. We define `WorkflowState` with fields for topic, data, cleaned data, processed data, and report.

In [2]:
class WorkflowState(TypedDict):
    topic: str
    data: str
    cleaned_data: str
    processed_data: str
    report: str


initial_state: WorkflowState = {
    "topic": "Machine Learning",
    "data": "",
    "cleaned_data": "",
    "processed_data": "",
    "report": ""
}

#### Create Multiple Nodes

Each node performs one task: collect, clean, process, and generate report. Each node reads the shared state and returns updated values.

In [3]:
def collect_data_node(state: WorkflowState):
    print("→ Node 1: Collecting data...")
    return {
        "data": f"Raw data collected on topic: {state['topic']}"
    }


def clean_data_node(state: WorkflowState):
    print("→ Node 2: Cleaning data...")
    return {
        "cleaned_data": f"Cleaned: {state['data'].replace('Raw', 'Clean')}"
    }


def process_data_node(state: WorkflowState):
    print("→ Node 3: Processing data...")
    return {
        "processed_data": f"Processed: {state['cleaned_data']} | 100% complete"
    }


def generate_report_node(state: WorkflowState):
    print("→ Node 4: Generating report...")
    return {
        "report": f"Report for '{state['topic']}':\n  {state['processed_data']}"
    }

#### Update State

Each node returns a partial dictionary. LangGraph automatically merges it into the shared state for the next node.

#### Connect Workflow

We add all nodes to the graph and connect them in sequence from START through collect → clean → process → report, then to END.

In [4]:
graph = StateGraph(WorkflowState)

graph.add_node("collect_data", collect_data_node)
graph.add_node("clean_data", clean_data_node)
graph.add_node("process_data", process_data_node)
graph.add_node("generate_report", generate_report_node)

graph.add_edge(START, "collect_data")
graph.add_edge("collect_data", "clean_data")
graph.add_edge("clean_data", "process_data")
graph.add_edge("process_data", "generate_report")
graph.add_edge("generate_report", END)

workflow = graph.compile()

#### Execute Graph

Invoke the compiled workflow with the initial state. The graph runs all four nodes in order, updating state at each step.

In [5]:
result = workflow.invoke(initial_state)

→ Node 1: Collecting data...
→ Node 2: Cleaning data...
→ Node 3: Processing data...
→ Node 4: Generating report...


#### Inspect Intermediate Outputs

Print each stage of the state to see how data flows between nodes.

In [6]:
print("== Final State ==")
for key, value in result.items():
    print(f"{key}: {value}")

== Final State ==
topic: Machine Learning
data: Raw data collected on topic: Machine Learning
cleaned_data: Cleaned: Clean data collected on topic: Machine Learning
processed_data: Processed: Cleaned: Clean data collected on topic: Machine Learning | 100% complete
report: Report for 'Machine Learning':
  Processed: Cleaned: Clean data collected on topic: Machine Learning | 100% complete


#### Test With a Different Topic

Re-run the same graph with new input to see the workflow generate a new report.

In [7]:
result2 = workflow.invoke({
    "topic": "Deep Learning",
    "data": "",
    "cleaned_data": "",
    "processed_data": "",
    "report": ""
})

print("== New Report ==")
print(result2["report"])

→ Node 1: Collecting data...
→ Node 2: Cleaning data...
→ Node 3: Processing data...
→ Node 4: Generating report...
== New Report ==
Report for 'Deep Learning':
  Processed: Cleaned: Clean data collected on topic: Deep Learning | 100% complete


#### Summary

Multiple nodes + shared state = multi-node workflow. Each node reads state, updates it, and passes it on to the next node in sequence.